<a href="https://colab.research.google.com/github/fourmodern/cellvit-tutorial/blob/main/02_CellViT_plusplus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CellViT++ 튜토리얼: 분류기 교체와 나만의 세포 분류기 학습

**CellViT++** (Hörst et al., arXiv:2501.05269, 2025)의 핵심 아이디어는 간단합니다.

```
  H&E ─► [CellViT 분할 모델 · 고정] ─► 세포 검출/분할 ───────────────┐
                 │                                                  ▼
                 └─► 세포별 토큰 임베딩 ─► [가벼운 MLP 분류기] ─► 세포 타입
                                            ▲ 이 부분만 교체·학습
```

- 무거운 분할 모델은 **그대로 두고**, 세포 임베딩 위의 작은 분류기(약 13만 파라미터)만 바꿔 끼웁니다.
- 분류기 학습에는 **세포 중심점(point) + 클래스** 라벨만 있으면 됩니다. 윤곽 마스크가 필요 없습니다.
- 학습은 GPU에서 수십 초~수 분이면 끝납니다.

| Part | 내용 |
|---|---|
| 1 | 사전학습된 분류기 7종 살펴보기 (Lizard, NuCLS, Ocelot, MIDOG, ...) |
| 2 | 같은 세포에 분류기를 바꿔 끼워 보기 (Python) |
| 3 | WSI 전체 추론 (`cellvit-inference` CLI) → 세포 JSON/GeoJSON/임베딩 그래프 |
| 4 | **점 라벨로 나만의 분류기 학습** (NuCLS 예제) → WSI에 적용 |

> CellViT 자체의 task(검출·분할·분류·임베딩)는 `01_CellViT_tasks.ipynb`를 먼저 보세요.

## 0. 환경 설정

**Colab 사용 시**
1. 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` 선택
2. 아래 설치 셀을 실행하면 패키지 설치 후 **런타임이 자동으로 재시작**됩니다 (cellvit이 `numpy<2`를 요구하기 때문).
3. 재시작 후 **처음부터 다시 실행**하세요. 설치 셀은 자동으로 건너뜁니다.

로컬 Jupyter에서는 `pip install cellvit openslide-bin umap-learn`이 된 커널이면 설치 셀이 아무것도 하지 않습니다.

In [ ]:
# 0-1. 설치 (Colab에서 최초 1회)
import sys, os, importlib.util

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and importlib.util.find_spec("cellvit") is None:
    !apt-get -qq install -y openslide-tools > /dev/null
    !pip -q install cellvit openslide-bin umap-learn
    print("설치 완료 → 런타임을 재시작합니다. 재시작 후 처음부터 다시 실행하세요.")
    os.kill(os.getpid(), 9)  # Colab 런타임 재시작

In [ ]:
# 0-2. 경로와 옵션
from pathlib import Path

WORKDIR = Path("/content/cellvit_tutorial") if IN_COLAB else Path.cwd()
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)
# 로컬 venv 커널에서도 cellvit-* CLI를 찾도록 현재 파이썬의 bin 경로를 PATH 앞에 추가
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]

# 모델 가중치(SAM-H 약 2.8GB)를 Google Drive에 캐시하면 다음 세션에서 재다운로드하지 않습니다.
USE_DRIVE_CACHE = False
if IN_COLAB and USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["CELLVIT_CACHE"] = "/content/drive/MyDrive/cellvit_cache"
else:
    os.environ["CELLVIT_CACHE"] = str(WORKDIR / "models")  # cellvit import 전에 설정해야 함

# "SAM"  : CellViT-SAM-H  (논문 최고 성능, encoder = SAM ViT-H, embedding 1280차원)
# "HIPT" : CellViT-256    (encoder = HIPT ViT-S/16, embedding 384차원, 가볍고 빠름)
MODEL_NAME = "SAM"
print("WORKDIR:", WORKDIR, "| CACHE:", os.environ["CELLVIT_CACHE"], "| MODEL:", MODEL_NAME)

In [ ]:
# 모델 가중치 + 예제 WSI 다운로드 (Zenodo, 처음 한 번만)
from cellvit.utils.cache_models import cache_cellvit_sam_h, cache_cellvit_256
from cellvit.utils.cache_test_database import cache_test_database

CKPT_PATH = cache_cellvit_sam_h() if MODEL_NAME == "SAM" else cache_cellvit_256()
cache_test_database(run_dir=str(WORKDIR))  # → WORKDIR/test_database (x40_svs, x20_svs, BRACS, ...)

!ls -lh {CKPT_PATH}
!ls test_database/*

# 사전학습 분류기 7종 (Zenodo classifier.zip)
from cellvit.utils.cache_models import cache_classifier
CLASSIFIER_DIR = cache_classifier() / ("sam-h" if MODEL_NAME == "SAM" else "vit256")

# Part 4용 NuCLS 점 라벨 예제 (CellViT++ GitHub repo에서 해당 폴더만 받기)
NUCLS_DIR = WORKDIR / "cellvitpp_repo/test_database/training_database/Example-Detection"
if not NUCLS_DIR.exists():
    !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/TIO-IKIM/CellViT-plus-plus.git cellvitpp_repo
    !cd cellvitpp_repo && git sparse-checkout set test_database/training_database/Example-Detection
!ls {CLASSIFIER_DIR} {NUCLS_DIR}

In [ ]:
# 공통 유틸 함수
import json, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import cv2
import matplotlib.pyplot as plt
import openslide

from cellvit.utils.tools import unflatten_dict
from cellvit.models.cell_segmentation.cellvit_sam import CellViTSAM
from cellvit.models.cell_segmentation.cellvit_256 import CellViT256

warnings.filterwarnings("ignore")

# 한글 폰트: 시스템에 NanumGothic이 없으면 Google Fonts에서 받아 등록
import matplotlib.font_manager as fm, urllib.request
_font = WORKDIR / "NanumGothic-Regular.ttf"
if not _font.exists():
    urllib.request.urlretrieve("https://github.com/google/fonts/raw/main/ofl/nanumgothic/NanumGothic-Regular.ttf", _font)
fm.fontManager.addfont(str(_font))
plt.rcParams["font.family"] = fm.FontProperties(fname=str(_font)).get_name()
plt.rcParams["axes.unicode_minus"] = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def load_cellvit(ckpt_path, device=DEVICE):
    """CellViT 체크포인트 → (model, config). cellvit.inference.CellViTInference._load_model과 동일한 방식."""
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    conf = unflatten_dict(ckpt["config"], ".")
    common = dict(
        num_nuclei_classes=conf["data"]["num_nuclei_classes"],
        num_tissue_classes=conf["data"]["num_tissue_classes"],
        regression_loss=conf["model"].get("regression_loss", False),
    )
    if ckpt["arch"] == "CellViTSAM":
        model = CellViTSAM(model_path=None, vit_structure=conf["model"]["backbone"], **common)
    else:
        model = CellViT256(model256_path=None, **common)
    model.load_state_dict(ckpt["model_state_dict"])
    return model.eval().to(device), conf


def to_tensor(img, conf):
    """RGB uint8 (H, W, 3) → 정규화된 (1, 3, H, W) 텐서"""
    norm = conf["transformations"].get("normalize", {})
    mean = np.array(norm.get("mean", (0.5, 0.5, 0.5)))
    std = np.array(norm.get("std", (0.5, 0.5, 0.5)))
    x = (img / 255.0 - mean) / std
    return torch.from_numpy(x.transpose(2, 0, 1)).float()[None]


@torch.inference_mode()
def run_cellvit(model, conf, img):
    """패치 하나에 대해 forward. H, W는 16의 배수여야 함 (40x, 0.25 µm/px 기준)."""
    x = to_tensor(img, conf).to(DEVICE)
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
        out = model(x, retrieve_tokens=True)
    return {k: v.float().cpu() for k, v in out.items()}


def postprocess(model, out, magnification=40):
    """NP/HV/NT 맵 → (instance map, 세포 리스트). 배경(type 0)으로 분류된 인스턴스는 제외."""
    pred = {
        "nuclei_binary_map": F.softmax(out["nuclei_binary_map"], dim=1),
        "nuclei_type_map": F.softmax(out["nuclei_type_map"], dim=1),
        "hv_map": out["hv_map"],
    }
    inst_map, cell_dicts = model.calculate_instance_map(pred, magnification=magnification)
    cells = [dict(c, id=int(i)) for i, c in cell_dicts[0].items() if c["type"] != 0]
    return inst_map[0].numpy().astype(np.int32), cells


def cell_embeddings(tokens, cells, token_size=16):
    """세포 bbox가 덮는 ViT 토큰을 평균 → 세포당 1개 벡터 (CellViT++와 동일한 방식)
    tokens: (D, H/16, W/16), bbox: [[row_min, col_min], [row_max, col_max]]"""
    embs = []
    for c in cells:
        bb = np.asarray(c["bbox"]) / token_size
        r0, c0 = np.floor(bb[0]).astype(int)
        r1, c1 = np.ceil(bb[1]).astype(int)
        embs.append(tokens[:, r0:r1, c0:c1].flatten(1).mean(1))
    return torch.stack(embs) if embs else torch.empty(0, tokens.shape[0])


TARGET_MPP = 0.25  # CellViT 학습 해상도 (40x)


def read_patch_40x(slide, x, y, size=1024):
    """level-0 좌표 (x, y)부터 0.25 µm/px 기준 size×size 패치를 읽음.
    20x(0.5 µm/px) 슬라이드라면 512×512를 읽어 2배 업샘플 → CellViT WSI 파이프라인과 같은 처리."""
    mpp = float(slide.properties.get("openslide.mpp-x", TARGET_MPP))
    scale = mpp / TARGET_MPP
    read = int(round(size / scale))
    im = np.array(slide.read_region((x, y), 0, (read, read)).convert("RGB"))
    if read != size:
        im = cv2.resize(im, (size, size), interpolation=cv2.INTER_CUBIC)
    return im, scale


def draw_cells(img, cells, colors, thickness=2):
    """세포 윤곽을 타입별 색으로 그림. colors: {type_id: (R, G, B)}"""
    canvas = img.copy()
    for c in cells:
        cnt = np.asarray(c["contour"], dtype=np.int32).reshape(-1, 1, 2)
        cv2.drawContours(canvas, [cnt], -1, colors.get(c["type"], (0, 0, 0)), thickness)
    return canvas


def legend(ax, names, colors, **kw):
    from matplotlib.patches import Patch
    handles = [Patch(color=np.array(colors[i]) / 255, label=n) for i, n in names.items() if i in colors]
    ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=9, **kw)

# 공통 시각화 도구 viz.py: 저장소에 있으면 그대로, 없으면(Colab) GitHub에서 받기
for _d in [WORKDIR, WORKDIR.parent]:
    if (_d / "viz.py").exists():
        sys.path.insert(0, str(_d)); break
else:
    urllib.request.urlretrieve("https://raw.githubusercontent.com/fourmodern/cellvit-tutorial/main/viz.py", WORKDIR / "viz.py")
    sys.path.insert(0, str(WORKDIR))
import viz

print("device:", DEVICE, "| torch", torch.__version__)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
# PanNuke 세포 타입 색상 (CellViT 논문 그림과 동일 계열)
PANNUKE_COLORS = {
    1: (255, 0, 0),     # Neoplastic   - 빨강
    2: (34, 221, 77),   # Inflammatory - 초록
    3: (35, 92, 236),   # Connective   - 파랑
    4: (254, 255, 0),   # Dead         - 노랑
    5: (255, 159, 68),  # Epithelial   - 주황
}

def palette(n):
    """분류기마다 클래스 수가 달라서 tab10 기반 색을 자동 생성"""
    cmap = plt.get_cmap("tab10")
    return {i: tuple(int(255 * v) for v in cmap(i % 10)[:3]) for i in range(n)}

## Part 1. 사전학습된 분류기 살펴보기

각 분류기는 `LinearClassifier` (임베딩 → hidden → 클래스 수)이며, 체크포인트 안에 클래스 이름(`label_map`)이 들어 있습니다.

In [ ]:
from cellvit.models.classifier.linear_classifier import LinearClassifier

def load_classifier(path):
    """CellViT++ 분류기 체크포인트 → (model, {id: 클래스명})"""
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    rc = unflatten_dict(ckpt["config"], ".")
    clf = LinearClassifier(
        embed_dim=ckpt["model_state_dict"]["fc1.weight"].shape[1],
        hidden_dim=rc["model"].get("hidden_dim", 100),
        num_classes=rc["data"]["num_classes"],
    )
    clf.load_state_dict(ckpt["model_state_dict"])
    return clf.eval(), {int(k): v for k, v in rc["data"]["label_map"].items()}

CLASSIFIERS = {}
for p in sorted(CLASSIFIER_DIR.glob("*.pth")):
    clf, labels = load_classifier(p)
    CLASSIFIERS[p.stem] = (clf, labels)
    n_par = sum(x.numel() for x in clf.parameters())
    print(f"{p.stem:12s} | {len(labels):2d} classes | {n_par/1e3:6.1f}K params | {list(labels.values())}")

## Part 2. 같은 세포, 다른 분류 체계

CellViT로 패치를 **한 번만** 추론해서 세포와 임베딩을 얻은 뒤, 분류기만 바꿔 가며 적용합니다. 분할 결과(윤곽)는 그대로이고 색(클래스)만 바뀌는 것을 볼 수 있습니다.

In [ ]:
model, conf = load_cellvit(CKPT_PATH)
PANNUKE_TYPES = {v: k for k, v in conf["dataset_config"]["nuclei_types"].items()}

# 01 노트북과 같은 패치: 피부 슬라이드(20x)의 표피-진피 경계 → 40x로 업샘플
WSI_PATH = WORKDIR / "test_database/x20_svs/CMU-1-Small-Region.svs"
slide = openslide.OpenSlide(str(WSI_PATH))
img, scale = read_patch_40x(slide, 768, 768, 1024)
thumb = np.array(slide.get_thumbnail((800, 800)).convert("RGB"))
ds = slide.dimensions[0] / thumb.shape[1]

out = run_cellvit(model, conf, img)
inst_map, cells = postprocess(model, out)
emb = cell_embeddings(out["tokens"][0], cells)
print(f"세포 {len(cells)}개, 임베딩 {tuple(emb.shape)}")

In [ ]:
@torch.no_grad()
def classify(clf, emb):
    p = F.softmax(clf(emb), dim=1)
    return p.argmax(1).numpy(), p.max(1).values.numpy()

SHOW = ["lizard", "nucls_main", "ocelot"]
zoom = (slice(256, 768), slice(256, 768))

fig, ax = plt.subplots(1, len(SHOW) + 1, figsize=(6 * (len(SHOW) + 1), 6.5))
ax[0].imshow(draw_cells(img, cells, PANNUKE_COLORS)[zoom])
ax[0].set_title("PanNuke (CellViT 기본 NT 디코더)")
legend(ax[0], {k: v for k, v in PANNUKE_TYPES.items() if k}, PANNUKE_COLORS)
for a, name in zip(ax[1:], SHOW):
    clf, labels = CLASSIFIERS[name]
    pred, _ = classify(clf, emb)
    relabeled = [dict(c, type=int(t)) for c, t in zip(cells, pred)]
    cols = palette(len(labels))
    a.imshow(draw_cells(img, relabeled, cols)[zoom])
    a.set_title(f"{name} 분류기")
    legend(a, labels, cols)
[a.axis("off") for a in ax]
plt.tight_layout(); plt.show()

Lizard는 림프구·형질세포·호중구·호산구까지 나누고, Ocelot은 종양/비종양 2분류만 합니다. **같은 임베딩**에서 목적에 맞는 체계를 고르면 되고, 원하는 체계가 없으면 Part 4처럼 직접 학습합니다.

### 인터랙티브: 버튼으로 분류 체계 바꿔 보기

위쪽 버튼으로 분류 체계를 바꾸면 **같은 세포 윤곽**에 색만 바뀝니다. `조직 사진만`을 누르면 오버레이 없이 H&E만 봅니다. 휠로 확대, 범례 클릭으로 클래스 켜고 끄기.

In [ ]:
layers = [("PanNuke", cells, lambda c: PANNUKE_COLORS[c["type"]], lambda c: PANNUKE_TYPES[c["type"]],
           lambda c: f"PanNuke: {PANNUKE_TYPES[c['type']]}", None)]
for name in ["lizard", "nucls_main", "panoptils", "ocelot"]:
    clf, labels = CLASSIFIERS[name]
    pred, conf_ = classify(clf, emb)
    cols_ = palette(len(labels))
    rel = [dict(c, type=int(t), conf=float(p)) for c, t, p in zip(cells, pred, conf_)]
    layers.append((name, rel, (lambda cols_: lambda c: cols_[c["type"]])(cols_),
                   (lambda labels: lambda c: labels[c["type"]])(labels),
                   (lambda labels, name: lambda c: f"{name}: {labels[c['type']]}<br>확률 {c['conf']:.2f}")(labels, name), None))
viz.interactive_layers(img, layers, title="같은 세포, 다른 분류 체계 (버튼으로 전환)")

## Part 3. WSI 전체 추론 (CLI)

`cellvit-inference`는 슬라이드를 1024px 패치로 나눠 추론하고, 패치 경계에서 중복된 세포를 정리해 슬라이드 단위 결과를 저장합니다.

| 옵션 | 의미 |
|---|---|
| `--model SAM \| HIPT` | 분할 모델 |
| `--nuclei_taxonomy` | `pannuke`(기본), `binary`, `lizard`, `consep`, `midog`, `nucls_main`, `nucls_super`, `ocelot`, `panoptils` |
| `--geojson` | QuPath용 GeoJSON 추가 저장 |
| `--graph` | 세포 임베딩 + 좌표를 `cells.pt`로 저장 (Part 4에서 사용) |
| `--cpu_count / --ray_worker / --ray_remote_cpus` | Colab 무료(2코어)에서는 작게 설정 |
| `process_wsi --wsi_mpp --wsi_magnification` | 메타데이터 없는 TIFF는 해상도를 직접 지정 |

예제는 01 노트북의 **피부 슬라이드(20x)** 전체입니다. mpp 메타데이터가 있어서 파이프라인이 자동으로 0.25 µm/px로 리샘플링하므로 `--wsi_mpp`를 줄 필요가 없습니다. Colab T4에서 수 분 걸립니다.

In [ ]:
# CLI는 별도 프로세스에서 모델을 다시 올리므로, 노트북이 잡고 있는 GPU 메모리를 먼저 비웁니다.
# (안 비우면 Colab T4에서 CUDA out of memory)
import gc
model = None; gc.collect(); torch.cuda.empty_cache()

OUTDIR = WORKDIR / "outputs/wsi_lizard"
OUTDIR.parent.mkdir(exist_ok=True)
CPU = 2 if IN_COLAB else min(8, os.cpu_count())

!cellvit-inference \
    --model {MODEL_NAME} \
    --nuclei_taxonomy lizard \
    --outdir {OUTDIR} \
    --geojson --graph \
    --cpu_count {CPU} --ray_worker 1 --ray_remote_cpus 1 \
    process_wsi \
    --wsi_path {WSI_PATH} > {OUTDIR}.log 2>&1

!tail -n 4 {OUTDIR}.log   # 전체 로그: outputs/wsi_lizard.log
!ls -lh {OUTDIR}/*

### 출력 파일 구조

| 파일 | 내용 |
|---|---|
| `cells.json` | 세포별 `contour`, `bbox`, `centroid`, `type`, `type_prob` + `type_map` |
| `cell_detection.json` | 중심 좌표와 타입만 (가벼움) |
| `cells.geojson` / `cell_detection.geojson` | QuPath에서 `File → Import objects` |
| `cells.pt` | `x`: 세포 임베딩 (N×D), `positions`: 좌표, `metadata` |

In [ ]:
slide_dir = next(p for p in OUTDIR.iterdir() if p.is_dir())
cells_json = json.loads((slide_dir / "cells.json").read_text())
wsi_cells = cells_json["cells"]
LIZARD = {int(k): v for k, v in cells_json["type_map"].items()}
graph = torch.load(slide_dir / "cells.pt", weights_only=False)

print("type_map:", LIZARD)
print(f"세포 수: {len(wsi_cells)} | graph.x: {tuple(graph.x.shape)} | graph.positions: {tuple(graph.positions.shape)}")

wsi_types = np.array([c["type"] for c in wsi_cells])
wsi_xy = np.array([c["centroid"] for c in wsi_cells])
cols = palette(len(LIZARD))

fig, ax = plt.subplots(1, 2, figsize=(18, 7), gridspec_kw={"width_ratios": [2, 1]})
ax[0].imshow(thumb)
for t, name in LIZARD.items():
    m = wsi_types == t
    if m.any():
        ax[0].scatter(wsi_xy[m, 0] / ds, wsi_xy[m, 1] / ds, s=3, color=np.array(cols[t]) / 255, label=name)
ax[0].legend(markerscale=4, loc="upper left", bbox_to_anchor=(1.01, 1)); ax[0].axis("off")
ax[0].set_title("WSI 전체 세포 (Lizard 분류)")
names, counts = zip(*[(LIZARD[t], (wsi_types == t).sum()) for t in LIZARD])
ax[1].barh(names, counts, color=[np.array(cols[t]) / 255 for t in LIZARD], ec="k")
ax[1].set_title("타입별 세포 수"); ax[1].invert_yaxis()
plt.tight_layout(); plt.show()

### 타입별 분포와 확신도

> ⚠️ cellvit 1.0.9에는 분류기를 쓸 때 `cells.json`의 `type_prob`가 `int()`로 잘려 **모두 0으로 저장되는 버그**가 있습니다(`postprocessing_numpy.py`). 그래서 확신도는 `cells.pt`의 임베딩에 같은 Lizard 분류기를 다시 적용해 직접 계산합니다.

In [ ]:
lz_clf, _ = CLASSIFIERS["lizard"]
lz_pred, lz_conf = classify(lz_clf, graph.x.float())
assert (lz_pred == wsi_types).all(), "저장된 타입과 재계산 타입이 같아야 함"
wsi_df = pd.DataFrame({
    "type": [LIZARD[t] for t in wsi_types], "confidence": lz_conf,
    "area_um2": [cv2.contourArea(np.asarray(c["contour"], np.float32)) * float(slide.properties["openslide.mpp-x"]) ** 2 for c in wsi_cells],
})
viz.type_summary(wsi_df, "type", {LIZARD[t]: cols[t] for t in LIZARD}, order=list(LIZARD.values()),
                 value_cols={"confidence": "분류기 확신도 (softmax 최대값)", "area_um2": "핵 면적 (µm²)"},
                 title="슬라이드 전체 세포 분포 (CellViT++ · Lizard)")

### 슬라이드 탐색

1. **전체 분포 (인터랙티브)**: 썸네일 위의 모든 세포. 확대하면 세포 위치가 보이고, 범례로 타입을 켜고 끌 수 있습니다.
2. **원본 해상도 탐색기**: 슬라이더로 위치와 영역 크기를 고르면 **원본 해상도 조직 사진**과 세포 윤곽을 나란히 보여 줍니다.

In [ ]:
liz_names = np.array([LIZARD[t] for t in wsi_types])
viz.wsi_view(thumb, ds, wsi_xy, liz_names, {LIZARD[t]: cols[t] for t in LIZARD}, order=list(LIZARD.values()),
             title="슬라이드 전체 세포 (Lizard) — 휠로 확대", marker_size=4)

In [ ]:
read_l0 = lambda x, y, w, h: np.array(slide.read_region((int(x), int(y)), 0, (int(w), int(h))).convert("RGB"))
viz.region_explorer(read_l0, wsi_xy, wsi_cells, lambda c: cols[c["type"]], lambda c: LIZARD[c["type"]],
                    [(LIZARD[t], cols[t]) for t in LIZARD], thumb, ds,
                    x_range=(0, slide.dimensions[0]), y_range=(0, slide.dimensions[1]), init=(1100, 1250),
                    sizes=(128, 256, 512, 1024))

## Part 4. 점 라벨로 나만의 세포 분류기 학습

**시나리오**: 병리 전문의가 세포 중심에 점을 찍고 클래스를 달아 준 데이터만 있을 때, 우리 연구에 맞는 분류 체계로 CellViT++를 맞춘다.

예제 데이터는 CellViT++ repo에 포함된 **NuCLS** (유방암, CC0) 일부입니다.
- train 10장 / test 5장, 256×256 px (40x)
- 라벨 CSV: `x, y, class` (점 라벨)
- 클래스 4종: Tumor / nonTIL Stromal / sTIL (기질 내 종양침윤림프구) / Other

**학습 과정**
1. 각 이미지를 CellViT에 넣어 세포 검출 + 임베딩 추출
2. 정답 점을 가장 가까운 검출 세포와 1:1 매칭 (헝가리안, 거리 ≤ 12px)
3. 매칭된 (임베딩, 클래스) 쌍으로 `LinearClassifier` 학습

In [ ]:
import pandas as pd
from PIL import Image
from scipy.optimize import linear_sum_assignment

NUCLS_LABELS = {0: "Tumor", 1: "nonTIL Stromal", 2: "sTIL", 3: "Other"}
MATCH_RADIUS = 12  # px (40x에서 약 3 µm)

def load_split(split):
    items = []
    for img_path in sorted((NUCLS_DIR / split / "images").glob("*.png")):
        lbl = pd.read_csv(NUCLS_DIR / split / "labels" / f"{img_path.stem}.csv", header=None, names=["x", "y", "label"])
        items.append((img_path.stem, np.array(Image.open(img_path).convert("RGB")), lbl))
    return items

def extract(items):
    """이미지별 CellViT 추론 → 정답 점과 검출 세포 매칭 → (임베딩, 라벨)"""
    X, y, n_gt = [], [], 0
    for name, im, lbl in items:
        o = run_cellvit(model, conf, im)
        _, det = postprocess(model, o)
        n_gt += len(lbl)
        if not det:
            continue
        e = cell_embeddings(o["tokens"][0], det)
        dc = np.array([c["centroid"] for c in det])              # (x, y)
        d = np.linalg.norm(lbl[["x", "y"]].values[:, None] - dc[None], axis=-1)
        gi, di = linear_sum_assignment(d)
        ok = d[gi, di] <= MATCH_RADIUS
        X.append(e[di[ok]]); y.append(lbl["label"].values[gi[ok]])
    X, y = torch.cat(X), torch.from_numpy(np.concatenate(y)).long()
    print(f"  정답 점 {n_gt}개 중 {len(y)}개 매칭 ({len(y)/n_gt:.0%})")
    return X, y

model, conf = load_cellvit(CKPT_PATH)  # Part 3에서 비웠던 모델 다시 로드
train_items, test_items = load_split("train"), load_split("test")
print("train"); X_tr, y_tr = extract(train_items)
print("test");  X_te, y_te = extract(test_items)
print("train 클래스 분포:", {NUCLS_LABELS[k]: int((y_tr == k).sum()) for k in NUCLS_LABELS})

In [ ]:
# 학습 데이터 예시: 점 라벨
cols4 = palette(4)
fig, ax = plt.subplots(1, 4, figsize=(20, 5.5))
for a, (name, im, lbl) in zip(ax, train_items[:4]):
    a.imshow(im)
    for k, n in NUCLS_LABELS.items():
        m = lbl["label"] == k
        a.scatter(lbl.x[m], lbl.y[m], s=40, color=np.array(cols4[k]) / 255, ec="w", label=n)
    a.set_title(name); a.axis("off")
ax[-1].legend(loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout(); plt.show()

### 분류기 학습

CellViT++ 공식 설정(`hidden_dim=256`, AdamW, dropout 0.1)과 비슷하게 맞추고, 클래스 불균형은 가중치로 보정합니다. 세포 수가 수백 개라 몇 초면 끝납니다.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score

def train_classifier(X, y, n_cls=4, hidden=256, epochs=300, lr=3e-4, seed=0):
    torch.manual_seed(seed)
    clf = LinearClassifier(embed_dim=X.shape[1], hidden_dim=hidden, num_classes=n_cls, drop_rate=0.1)
    w = 1.0 / torch.bincount(y, minlength=n_cls).clamp(min=1).float()
    loss_fn = torch.nn.CrossEntropyLoss(weight=w / w.sum() * n_cls)
    opt = torch.optim.AdamW(clf.parameters(), lr=lr, weight_decay=1e-4)
    losses = []
    for _ in range(epochs):  # 데이터가 작아서 full-batch
        clf.train(); opt.zero_grad()
        loss = loss_fn(clf(X), y); loss.backward(); opt.step()
        losses.append(loss.item())
    return clf.eval(), losses

t = time.time()
my_clf, losses = train_classifier(X_tr, y_tr)
print(f"학습 시간: {time.time() - t:.1f}s")

pred, pred_conf = classify(my_clf, X_te)
print(classification_report(y_te, pred, labels=list(NUCLS_LABELS), target_names=list(NUCLS_LABELS.values()), zero_division=0))

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(losses); ax.set_xlabel("epoch"); ax.set_ylabel("train loss"); ax.set_title("학습 곡선")
plt.tight_layout(); plt.show()

### 정확도 시각화 (test 세포 = 정답 있음)

- **클래스별 precision / recall / F1**: 막대 아래 n은 test 세포 수. 점선은 macro-F1.
- **혼동행렬**: 각 정답 클래스(행)가 어떤 클래스로 예측됐는지 비율(괄호 안은 세포 수).
- **확신도 vs 정답률**: 모델이 확신할수록 실제로 더 맞는지. 대각선에 가까울수록 확신도를 믿을 수 있습니다.

In [ ]:
viz.classification_metrics(y_te.numpy(), pred, NUCLS_LABELS, cols4, title="직접 학습한 분류기 — NuCLS test 세포")
viz.confidence_vs_accuracy(pred_conf, pred == y_te.numpy(), bins=5)

### 라벨이 얼마나 필요할까?

학습 세포 수를 줄여 가며 test macro-F1을 봅니다(각 크기별 5회 반복). CellViT++ 논문의 주장대로 **적은 수의 점 라벨로도** 성능이 빠르게 올라가는지 확인합니다.

> 예제 데이터는 train 약 400개, test 약 190개 세포뿐이라 수치 자체는 변동이 큽니다. 경향만 보세요.

In [ ]:
rng = np.random.default_rng(0)
sizes = [25, 50, 100, 200, len(y_tr)]
res = []
for n in sizes:
    for r in range(5):
        idx = rng.choice(len(y_tr), n, replace=False)
        c, _ = train_classifier(X_tr[idx], y_tr[idx], seed=r)
        res.append((n, f1_score(y_te, classify(c, X_te)[0], average="macro")))
res = pd.DataFrame(res, columns=["n_train", "macro_f1"])
summary = res.groupby("n_train").macro_f1.agg(["mean", "std"])
print(summary.round(3))

fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(summary.index, summary["mean"], yerr=summary["std"], marker="o", capsize=4)
ax.set_xscale("log"); ax.set_xlabel("학습에 쓴 세포 수 (점 라벨)"); ax.set_ylabel("test macro-F1")
ax.set_title("라벨 수에 따른 성능"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### test 이미지에서 예측 확인

In [ ]:
fig, ax = plt.subplots(2, len(test_items), figsize=(4.2 * len(test_items), 8.5))
for j, (name, im, lbl) in enumerate(test_items):
    o = run_cellvit(model, conf, im)
    _, det = postprocess(model, o)
    p, _ = classify(my_clf, cell_embeddings(o["tokens"][0], det))
    ax[0, j].imshow(im)
    for k in NUCLS_LABELS:
        m = lbl["label"] == k
        ax[0, j].scatter(lbl.x[m], lbl.y[m], s=40, color=np.array(cols4[k]) / 255, ec="w")
    ax[1, j].imshow(draw_cells(im, [dict(c, type=int(t)) for c, t in zip(det, p)], cols4, thickness=1))
    ax[0, j].set_title(f"{name}: 정답 (점)"); ax[1, j].set_title("예측 (윤곽)")
[a.axis("off") for a in ax.ravel()]
legend(ax[0, -1], NUCLS_LABELS, cols4)
plt.tight_layout(); plt.show()

### 학습한 분류기를 저장하고 WSI 전체에 적용

CellViT++ 체크포인트와 같은 형식(`model_state_dict` + 평탄화된 `config`)으로 저장합니다. 이 파일은 CellViT++ repo의 `detect_cells.py --classifier_path`에 그대로 넣을 수 있습니다.

여기서는 Part 3에서 저장한 **WSI 전체 세포 임베딩(`cells.pt`)** 에 바로 적용합니다. 분할 모델을 다시 돌릴 필요가 없다는 점이 CellViT++ 방식의 장점입니다.

> ⚠️ 이 분류기는 **유방암**(NuCLS) 세포로 학습했는데 예제 WSI는 **정상 피부**입니다. 결과는 '분류기를 WSI에 붙이는 방법'의 시연일 뿐이며, 실제로는 학습 데이터와 같은 도메인의 슬라이드에 써야 합니다.

In [ ]:
ckpt = {
    "model_state_dict": my_clf.state_dict(),
    "config": {"data.num_classes": 4, "data.label_map": NUCLS_LABELS, "model.hidden_dim": 256},
}
my_path = WORKDIR / "outputs" / "my_nucls_classifier.pth"
torch.save(ckpt, my_path)

clf_reloaded, labels = load_classifier(my_path)  # Part 1과 같은 로더로 다시 읽힘
wsi_pred, wsi_conf = classify(clf_reloaded, graph.x.float())

fig, ax = plt.subplots(1, 2, figsize=(18, 7), gridspec_kw={"width_ratios": [2, 1]})
ax[0].imshow(thumb)
pos = graph.positions.numpy()
for k, n in labels.items():
    m = wsi_pred == k
    ax[0].scatter(pos[m, 0] / ds, pos[m, 1] / ds, s=3, color=np.array(cols4[k]) / 255, label=n)
ax[0].legend(markerscale=4, loc="upper left", bbox_to_anchor=(1.01, 1)); ax[0].axis("off")
ax[0].set_title("WSI 전체 세포 - 직접 학습한 NuCLS 분류기")
ax[1].barh(list(labels.values()), [(wsi_pred == k).sum() for k in labels],
           color=[np.array(cols4[k]) / 255 for k in labels], ec="k")
ax[1].invert_yaxis(); ax[1].set_title("타입별 세포 수")
plt.tight_layout(); plt.show()

In [ ]:
viz.type_summary(pd.DataFrame({"type": [labels[k] for k in wsi_pred], "confidence": wsi_conf}), "type",
                 {labels[k]: cols4[k] for k in labels}, order=list(labels.values()),
                 value_cols={"confidence": "분류기 확신도"}, title="피부 슬라이드에 적용한 NuCLS 분류기의 분포")

### 직접 학습한 분류기가 각 클래스로 부른 세포들

슬라이드 전체 세포를 학습한 분류기의 클래스별로 모아, 원본 조직에서 잘라 보여 줍니다. (정상 피부라 `Tumor`로 분류된 세포는 대부분 표피 세포일 것입니다 — 도메인 차이를 눈으로 확인)

In [ ]:
my_cells = [dict(c, type=int(t)) for c, t in zip(wsi_cells, wsi_pred)]
viz.cell_gallery(my_cells, lambda c: labels[c["type"]], lambda c: cols4[c["type"]], order=list(labels.values()),
                 get_crop=lambda x, y, s: read_l0(x, y, s, s), n=10, size=48,
                 title="직접 학습한 NuCLS 분류기의 클래스별 세포 (피부 슬라이드, 20x)")

## 정리

| | CellViT | CellViT++ |
|---|---|---|
| 분류 체계 | PanNuke 5종 고정 | 분류기 교체로 자유롭게 |
| 새 체계 학습 | 마스크 라벨 + 전체 재학습 | **점 라벨 + MLP만 학습 (수 초~수 분)** |
| 재추론 | 필요 | 저장된 임베딩(`cells.pt`)에 분류기만 적용 |

**실전에서 더 해 볼 것**
- 실제 프로젝트 데이터는 수천 개 이상의 점 라벨과 교차검증으로 학습하세요. 공식 학습 스크립트: CellViT++ repo의 `cellvit/train_cell_classifier_head.py`
- 점 라벨링은 CellViT++ repo의 웹 기반 annotation tool이나 QuPath로 할 수 있습니다.
- `cells.pt`의 임베딩과 좌표로 세포 그래프(GNN), 이웃 조성 분석, TIL 밀도 같은 공간 지표를 만들 수 있습니다.
- mpp 메타데이터가 없는 TIFF는 `process_wsi --wsi_mpp 0.25 --wsi_magnification 40`처럼 해상도를 직접 지정합니다.
- 라이선스: Apache 2.0 + Commons Clause → 상업적 이용 시 저자 허가 필요